In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sys 
import os
import seaborn as sns
import matplotlib.ticker as ticker
from tqdm import tqdm
import pickle
from pathlib import Path

pythonPackagePath = os.path.abspath(r'D:\LLSM-CME-ANALYSIS\Final\src')
sys.path.append(pythonPackagePath)

Analyzing trackability scores from randomly defined DynROIs (parameter sweep)

In [ ]:
def load_all_tracking_data():
    base_path = Path(r"Z:\Abhi\LLSM_analysis\controlOS_analysis\dynROI_tracking_trackability")
    data = {}
    
    # Get all parameter folders that start with msn_
    param_folders = [f for f in os.listdir(base_path) if f.startswith('msn_')]
    
    for param_folder in param_folders:
        data[param_folder] = {}
        param_path = base_path / param_folder
        
        # Check if parameter folder exists and get ROI folders
        if param_path.exists():
            roi_folders = [f for f in os.listdir(param_path) if f.startswith('roi_')]
            
            for roi_folder in roi_folders:
                pkl_path = param_path / roi_folder / "trackInROI" / "Channel_1_tracking_result.pkl"
                
                # Only load if the pickle file exists
                if pkl_path.exists():
                    try:
                        with open(pkl_path, 'rb') as f:
                            data[param_folder][roi_folder] = pickle.load(f)
                    except Exception as e:
                        print(f"Warning: Could not load {pkl_path}: {e}")
                        data[param_folder][roi_folder] = None
    
    return data

tracking_sweep_data = load_all_tracking_data()

In [ ]:
# trial = tracking_sweep_data['msn_gap2_rad1']['roi_003']
# trial[trial['track_id'] == 856]

In [ ]:
def create_trackability_dataframe(tracking_data):
    data_list = []
    
    for param, roi_dict in tracking_data.items():
        # Skip empty parameter dictionaries
        if not roi_dict:
            continue
            
        for roi, df in roi_dict.items():
            if df is not None and 'segTrackability' in df.columns:
                # Keep all trackability values as a list/array
                trackability_values = df['segTrackability'].values.tolist()  # or .tolist()
                data_list.append({
                    'parameter': param,
                    'roi': roi,
                    'trackability': trackability_values
                })
    
    # Create DataFrame and set multi-index
    df = pd.DataFrame(data_list)
    trackability = df.set_index(['parameter', 'roi'])
    
    return trackability

trackability = create_trackability_dataframe(tracking_sweep_data)

In [ ]:
trackability


In [ ]:
# User configuration
PARAM_TYPE = 'rad'  # Change to 'gap' or 'rad' as needed
PARAM_VALUES = ['rad1', 'rad2', 'rad3']  # Use None for defaults, or specify like ['gap1', 'gap2'] or ['rad1', 'rad2', 'rad3']
FIXED_PARAMS = {'gap': 'gap1', 'merge_split': 'no'} # Set fixed parameters (e.g., {'rad': 'rad2'} for gap sweep, {'gap': 'gap3'} for rad sweep)
ROIS = None  # Use None for all ROIs, or specify like ['roi_001', 'roi_002']

def create_plot_data(trackability, param_type='gap', param_values=None, 
                     fixed_params=None, rois=None):
    """
    Create plot data for trackability analysis.
    
    Parameters:
    -----------
    param_type : str, 'gap' or 'rad'
        Which parameter to analyze (sweep)
    param_values : list, optional
        Specific parameter values to include (e.g., ['gap1', 'gap2', 'gap3'] or ['rad1', 'rad2'])
    fixed_params : dict, optional
        Fixed parameter values (e.g., {'rad': 'rad1'} when sweeping gap, or {'gap': 'gap2'} when sweeping rad)
    rois : list, optional
        Specific ROIs to include (default: ['roi_001', 'roi_002', 'roi_003'])
    """
    plot_data = []
    
    # Set defaults
    if rois is None:
        rois = ['roi_001', 'roi_002', 'roi_003']
    
    # Determine which parameters to iterate over based on param_type
    if param_type == 'gap':
        if param_values is None:
            param_values = ['gap1', 'gap2', 'gap3']
        if fixed_params is None:
            fixed_params = {'rad': 'rad1'}  # Default fixed radius
        fixed_rad = fixed_params.get('rad', 'rad1')
        params = [f'msn_{gap}_{fixed_rad}' for gap in param_values]
        extract_key = lambda p: p.split('_')[1]  # Extract gap1, gap2, gap3
    elif param_type == 'rad':
        if param_values is None:
            param_values = ['rad1', 'rad2', 'rad3']
        if fixed_params is None:
            fixed_params = {'gap': 'gap1'}  # Default fixed gap
        fixed_gap = fixed_params.get('gap', 'gap1')
        params = [f'msn_{fixed_gap}_{rad}' for rad in param_values]
        extract_key = lambda p: p.split('_')[2]  # Extract rad1, rad2
    else:
        raise ValueError("param_type must be 'gap' or 'rad'")
    
    for param in params:
        for roi in rois:
            if (param, roi) in trackability.index:
                values = trackability.loc[param, roi].iloc[0]  # This is already the list
                param_label = extract_key(param)
                
                for value in values:
                    plot_data.append({
                        'Parameter': param_label,
                        'Trackability': value,
                        'ROI': roi
                    })
    
    return pd.DataFrame(plot_data), fixed_params



# Create the plot data
plot_df, used_fixed_params = create_plot_data(trackability, 
                                              param_type=PARAM_TYPE, 
                                              param_values=PARAM_VALUES, 
                                              fixed_params=FIXED_PARAMS,
                                              rois=ROIS)

In [ ]:
plot_df

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

def get_fixed_param_string(param_type, fixed_params):
    """Generate a string describing the fixed parameters."""
    parts = []
    
    # Add merge/split info if provided
    if 'merge_split' in fixed_params:
        parts.append(f"merge/split = {fixed_params['merge_split']}")
    
    # Add the other fixed parameter
    if param_type == 'gap':
        rad_val = fixed_params.get('rad', 'rad1')
        rad_num = rad_val.replace('rad', '')
        parts.append(f"min_radius = {rad_num}")
    else:  # param_type == 'rad'
        gap_val = fixed_params.get('gap', 'gap1')
        gap_num = gap_val.replace('gap', '')
        parts.append(f"gap_size = {gap_num}")
    
    return ', '.join(parts)

# Configuration for plot titles
PLOT_CONFIGS = {
    'gap': {
        'xlabel': 'Gap Size',
        'legend_title': 'Region of Interest',
        'title_prefix': 'Gap Size Parameter Sweep'
    },
    'rad': {
        'xlabel': 'Minimum Radius',
        'legend_title': 'Region of Interest',
        'title_prefix': 'Minimum Radius Parameter Sweep'
    }
}

# Get configuration based on param_type
config = PLOT_CONFIGS[PARAM_TYPE]
fixed_param_str = get_fixed_param_string(PARAM_TYPE, used_fixed_params)
full_title = f"{config['title_prefix']}\n({fixed_param_str})"
# Set style and color palette
sns.set_style("whitegrid")
plt.figure(figsize=(10, 7))

# Create violin plot with better styling
ax = sns.violinplot(x="Parameter", y="Trackability", data=plot_df, 
                    inner=None, color='lightblue', alpha=0.6,
                    linewidth=1.5, saturation=0.8)

# Calculate means for each ROI
means_df = plot_df.groupby(['Parameter', 'ROI'])['Trackability'].mean().reset_index()

# Create a custom color palette for ROIs
roi_colors = ['#1f77b4', '#ff7f0e', '#2ca02c']  # Blue, Orange, Green
unique_rois = sorted(plot_df['ROI'].unique())
roi_palette = dict(zip(unique_rois, roi_colors[:len(unique_rois)]))

# Plot means as larger points with better styling
sns.stripplot(x="Parameter", y="Trackability", hue="ROI", data=means_df, 
              size=12, dodge=False, jitter=True, palette=roi_palette,
              edgecolor='white', linewidth=2, alpha=0.9)

# Improve formatting
plt.title(full_title, fontsize=16, fontweight='bold', pad=20)
plt.ylabel('Trackability Score', fontsize=14, fontweight='semibold')
plt.xlabel(config['xlabel'], fontsize=14, fontweight='semibold')

# Improve legend
handles, labels = ax.get_legend_handles_labels()
new_labels = [f'DynROI {i+1}' for i in range(len(unique_rois))]
plt.legend(handles, new_labels, title=config['legend_title'], 
           title_fontsize=12, fontsize=11, loc='lower right',
           frameon=True, fancybox=True, shadow=True)

# Set y-axis limits for better visibility
plt.ylim(0, 1.05)

# Improve axis labels
plt.xticks(fontsize=12)
plt.yticks(fontsize=11)

# Add subtle grid
plt.grid(True, alpha=0.3, linestyle='--')

# Tight layout for better spacing
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Set bin edges
bins = np.arange(-0.1, 1.15, 0.05)  # 0.05 bin size, upper limit 1.1 inclusive

# Filter out NaN values and round
clean_df = plot_df.dropna(subset=['Trackability']).round(4)

# Set style
sns.set_style("whitegrid")

# Create histogram
plt.figure(figsize=(10, 6))
ax = sns.histplot(data=clean_df, x='Trackability', bins=bins, hue='Parameter',
             multiple='dodge', shrink=0.85, palette='pastel', edgecolor='black')

# Dynamic title based on parameter type and fixed parameters
hist_title = f"Histogram of Trackability Scores by {config['xlabel']}\n({fixed_param_str})"
plt.title(hist_title, fontsize=15, fontweight='bold')
plt.xlabel("Trackability Score", fontsize=13)
plt.ylabel("Count", fontsize=13)
plt.xticks(bins)
plt.grid(True, linestyle='--', alpha=0.3)

# Update the existing legend title (created by histplot)
legend = ax.get_legend()
if legend:
    legend.set_title(config['xlabel'])
    legend.get_title().set_fontsize(11)
    for text in legend.get_texts():
        text.set_fontsize(10)

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd

# Set bin edges and labels
bins = np.arange(-0.1, 1.15, 0.05)  # 0.05 bin size, upper limit 1.1 inclusive
bin_labels = [f"({bins[i]:.2f}, {bins[i+1]:.2f}]" for i in range(len(bins)-1)]

# Create clean copy and snap values to 4 decimal places
clean_df = plot_df.dropna(subset=['Trackability']).copy()
clean_df['Trackability'] = clean_df['Trackability'].round(4)

# Bin the data
clean_df.loc[:, 'bin'] = pd.cut(clean_df['Trackability'], bins=bins, labels=bin_labels, right=True)
    
# Count per bin and parameter value
bin_counts = clean_df.groupby(['Parameter', 'bin'], observed=True).size().reset_index(name='count')

# Total counts per parameter value for normalization
param_totals = bin_counts.groupby('Parameter')['count'].transform('sum')
bin_counts['percent'] = 100 * bin_counts['count'] / param_totals

# Create the plot
plt.figure(figsize=(14, 6))
ax = sns.barplot(data=bin_counts, x='bin', y='percent', hue='Parameter', palette='pastel')

# Dynamic title based on parameter type and fixed parameters
bar_title = f"Percentage of Trackability Scores per Bin by {config['xlabel']}\n({fixed_param_str})"
ax.set_title(bar_title, fontsize=16, fontweight='bold', pad=15)
ax.set_xlabel("Trackability Score Bin", fontsize=14, fontweight='semibold', labelpad=10)
ax.set_ylabel("Percentage (%)", fontsize=14, fontweight='semibold', labelpad=10)

# Tick formatting
plt.xticks(rotation=90, fontsize=11)
plt.yticks(fontsize=11)

# Grid
plt.grid(axis='y', linestyle='--', alpha=0.4)

# Legend with dynamic title
plt.legend(title=config['xlabel'], title_fontsize=12, fontsize=11, 
           loc='upper left', frameon=True, shadow=False)

# Layout
plt.tight_layout()
plt.show()

In [ ]:
# Print number of segments per bin per Parameter
print(f"🔢 Segment Count per Bin per {config['xlabel']}:\n")
for param in bin_counts['Parameter'].unique():
   print(f"{config['xlabel']} = {param}")
   subset = bin_counts[bin_counts['Parameter'] == param][['bin', 'count']]
   for _, row in subset.iterrows():
       print(f"  Bin {row['bin']}: {int(row['count'])} segments")
   total = subset['count'].sum()
   print(f"  ➤ Total segments for {config['xlabel']} {param}: {int(total)}\n")

# Print total segment count across all parameters
overall_total = bin_counts['count'].sum()
print(f"✅ Overall total number of segments: {int(overall_total)}")

In [ ]:
clean_df

Analyzing trackability score distributions per track

In [ ]:
# % of tracks with a segment having TS < 0.95 (0.9 and under)
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

def calculate_low_trackability_percentage(tracking_sweep_data, param_type='gap', 
                                          param_values=None, fixed_params=None, 
                                          rois=None, threshold=0.95):
    """
    Calculate percentage of tracks with any segment having trackability < threshold.
    """
    plot_data = []
    
    # Set defaults
    if rois is None:
        rois = ['roi_001', 'roi_002', 'roi_003']
    
    # Determine which parameters to iterate over based on param_type
    if param_type == 'gap':
        if param_values is None:
            param_values = ['gap1', 'gap2', 'gap3']
        if fixed_params is None:
            fixed_params = {'rad': 'rad1'}
        fixed_rad = fixed_params.get('rad', 'rad1')
        params = [(f'msn_{gap}_{fixed_rad}', gap) for gap in param_values]
    elif param_type == 'rad':
        if param_values is None:
            param_values = ['rad1', 'rad2', 'rad3']
        if fixed_params is None:
            fixed_params = {'gap': 'gap1'}
        fixed_gap = fixed_params.get('gap', 'gap1')
        params = [(f'msn_{fixed_gap}_{rad}', rad) for rad in param_values]
    else:
        raise ValueError("param_type must be 'gap' or 'rad'")
    
    for param_key, param_label in params:
        for roi in rois:
            if param_key in tracking_sweep_data and roi in tracking_sweep_data[param_key]:
                df = tracking_sweep_data[param_key][roi]
                
                # Group by track_id and check if any segment has trackability < threshold
                tracks_with_low = df.groupby('track_id')['segTrackability'].apply(
                    lambda x: (x < threshold).any()
                )
                
                # Calculate percentage
                percent_low = 100 * tracks_with_low.sum() / len(tracks_with_low)
                
                plot_data.append({
                    'Parameter': param_label,
                    'ROI': roi,
                    'Percentage': percent_low
                })
    
    return pd.DataFrame(plot_data), fixed_params

# User configuration (same as before)
PARAM_TYPE = 'rad'  # or 'gap'
PARAM_VALUES = None  # Use all available values
FIXED_PARAMS = {'gap': 'gap1', 'merge_split': 'no'}
ROIS = None
THRESHOLD = 0.80  # Trackability threshold

# Calculate the percentages
percentage_df, used_fixed_params = calculate_low_trackability_percentage(
    tracking_sweep_data, 
    param_type=PARAM_TYPE,
    param_values=PARAM_VALUES,
    fixed_params=FIXED_PARAMS,
    rois=ROIS,
    threshold=THRESHOLD
)

# Create the plot
sns.set_style("whitegrid")
plt.figure(figsize=(10, 6))

# Create bar plot with ROIs as separate bars for each parameter
ax = sns.barplot(data=percentage_df, x='Parameter', y='Percentage', hue='ROI',
                 palette='pastel', edgecolor='black', linewidth=1.5)

# Get configuration and title
config = PLOT_CONFIGS[PARAM_TYPE]
fixed_param_str = get_fixed_param_string(PARAM_TYPE, used_fixed_params)
title = f"Percentage of Tracks with Any Segment Having Trackability < {THRESHOLD}\n({fixed_param_str})"

plt.title(title, fontsize=14, fontweight='bold', pad=15)
plt.xlabel(config['xlabel'], fontsize=12, fontweight='semibold')
plt.ylabel(f'Percentage of Tracks (%)', fontsize=12, fontweight='semibold')

# Improve legend
handles, labels = ax.get_legend_handles_labels()
new_labels = [f'DynROI {i+1}' for i in range(len(labels))]
plt.legend(handles, new_labels, title='Region of Interest', 
           title_fontsize=11, fontsize=10, loc='best',
           frameon=True, fancybox=True, shadow=True)

# Add horizontal line at specific percentages for reference
# plt.axhline(y=50, color='gray', linestyle='--', alpha=0.5, linewidth=1)
# plt.axhline(y=25, color='gray', linestyle='--', alpha=0.3, linewidth=1)
# plt.axhline(y=75, color='gray', linestyle='--', alpha=0.3, linewidth=1)

# Set y-axis limits
plt.ylim(0, 100)

# Add value labels on top of bars
for container in ax.containers:
    ax.bar_label(container, fmt='%.1f%%', padding=3, fontsize=9)

# Grid
plt.grid(True, axis='y', alpha=0.3, linestyle='--')

plt.tight_layout()
plt.show()

# Print summary statistics in a table format
print(f"\n📊 Summary: Tracks with Trackability < {THRESHOLD}")
print("=" * 50)

# Create a pivot table for easy copying
pivot_df = percentage_df.pivot(index='ROI', columns='Parameter', values='Percentage')
pivot_df.index = [f'DynROI {int(idx.split("_")[-1])}' for idx in pivot_df.index]

# Print as tab-separated values for easy Excel paste
print("\nTab-separated format (copy and paste to Excel):")
print(f"{config['xlabel']}\t" + "\t".join(pivot_df.columns))
for idx, row in pivot_df.iterrows():
   values = "\t".join([f"{v:.2f}" for v in row])
   print(f"{idx}\t{values}")

# Add means row
means = pivot_df.mean()
mean_values = "\t".join([f"{v:.2f}" for v in means])
print(f"Mean\t{mean_values}")

# Also print as CSV format
print("\nCSV format:")
print(f"{config['xlabel']}," + ",".join(pivot_df.columns))
for idx, row in pivot_df.iterrows():
   values = ",".join([f"{v:.2f}" for v in row])
   print(f"{idx},{values}")
mean_values_csv = ",".join([f"{v:.2f}" for v in means])
print(f"Mean,{mean_values_csv}")

# Also print as Markdown table
print("\nMarkdown table format:")
print(f"| {config['xlabel']} | " + " | ".join(pivot_df.columns) + " |")
print("|" + "---|" * (len(pivot_df.columns) + 1))
for idx, row in pivot_df.iterrows():
   values = " | ".join([f"{v:.2f}" for v in row])
   print(f"| {idx} | {values} |")
print(f"| **Mean** | " + " | ".join([f"**{v:.2f}**" for v in means]) + " |")

In [ ]:
# Calculate average trackability per track
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

def calculate_avg_trackability_per_track(tracking_sweep_data, param_type='gap', 
                                         param_values=None, fixed_params=None, 
                                         rois=None):
    """
    Calculate average trackability score for each track.
    """
    plot_data = []
    
    # Set defaults
    if rois is None:
        rois = ['roi_001', 'roi_002', 'roi_003']
    
    # Determine which parameters to iterate over based on param_type
    if param_type == 'gap':
        if param_values is None:
            param_values = ['gap1', 'gap2', 'gap3']
        if fixed_params is None:
            fixed_params = {'rad': 'rad1'}
        fixed_rad = fixed_params.get('rad', 'rad1')
        params = [(f'msn_{gap}_{fixed_rad}', gap) for gap in param_values]
    elif param_type == 'rad':
        if param_values is None:
            param_values = ['rad1', 'rad2', 'rad3']
        if fixed_params is None:
            fixed_params = {'gap': 'gap1'}
        fixed_gap = fixed_params.get('gap', 'gap1')
        params = [(f'msn_{fixed_gap}_{rad}', rad) for rad in param_values]
    else:
        raise ValueError("param_type must be 'gap' or 'rad'")
    
    for param_key, param_label in params:
        for roi in rois:
            if param_key in tracking_sweep_data and roi in tracking_sweep_data[param_key]:
                df = tracking_sweep_data[param_key][roi]
                
                # Filter out NaN values
                df_valid = df[df['segTrackability'].notna()]
                
                # Calculate average trackability per track
                track_avgs = df_valid.groupby('track_id')['segTrackability'].mean()
                
                # Add each track's average to plot_data
                for track_id, avg_score in track_avgs.items():
                    plot_data.append({
                        'Parameter': param_label,
                        'ROI': roi,
                        'Avg_Trackability': avg_score,
                        'track_id': track_id
                    })
    
    return pd.DataFrame(plot_data), fixed_params

In [ ]:
# User configuration
PARAM_TYPE = 'rad'  # or 'gap'
PARAM_VALUES = None  # Use all available values
FIXED_PARAMS = {'gap': 'gap1', 'merge_split': 'no'}
ROIS = None

# Calculate the average trackability scores
avg_df, used_fixed_params = calculate_avg_trackability_per_track(
    tracking_sweep_data, 
    param_type=PARAM_TYPE,
    param_values=PARAM_VALUES,
    fixed_params=FIXED_PARAMS,
    rois=ROIS
)

# Create the boxplot
sns.set_style("whitegrid")
plt.figure(figsize=(10, 7))

# Prepare data with better ROI labels
avg_df_plot = avg_df.copy()
avg_df_plot['ROI_Label'] = avg_df_plot['ROI'].apply(lambda x: f"DynROI {int(x.split('_')[-1])}")

# Create boxplot grouped by Parameter with ROIs as hue
ax = sns.boxplot(data=avg_df_plot, x='Parameter', y='Avg_Trackability', hue='ROI_Label',
                 palette='pastel', linewidth=1.5)

# Get configuration and title
config = PLOT_CONFIGS[PARAM_TYPE]
fixed_param_str = get_fixed_param_string(PARAM_TYPE, used_fixed_params)
title = f"Distribution of Average Trackability Scores per Track\n({fixed_param_str})"

plt.title(title, fontsize=14, fontweight='bold', pad=15)
plt.xlabel(config['xlabel'], fontsize=12, fontweight='semibold')
plt.ylabel('Average Trackability Score per Track', fontsize=12, fontweight='semibold')

# Improve legend
plt.legend(title='Region of Interest', title_fontsize=11, fontsize=10, 
           loc='lower right', frameon=True, fancybox=True, shadow=True)

# Set y-axis limits
plt.ylim(0, 1.05)

# # Grid
plt.grid(False)

plt.tight_layout()
plt.show()

# # Print summary statistics
# print(f"\n📊 Summary Statistics: Average Trackability per Track")
# print("=" * 60)

# for param in avg_df['Parameter'].unique():
#     print(f"\n{config['xlabel']}: {param}")
#     param_data = avg_df[avg_df['Parameter'] == param]
    
#     for roi in sorted(param_data['ROI'].unique()):
#         roi_data = param_data[param_data['ROI'] == roi]['Avg_Trackability']
#         roi_num = int(roi.split('_')[-1])
#         print(f"\n  DynROI {roi_num}:")
#         print(f"    Mean:   {roi_data.mean():.4f}")
#         print(f"    Median: {roi_data.median():.4f}")
#         print(f"    Std:    {roi_data.std():.4f}")
#         print(f"    Min:    {roi_data.min():.4f}")
#         print(f"    Max:    {roi_data.max():.4f}")
#         print(f"    Q1:     {roi_data.quantile(0.25):.4f}")
#         print(f"    Q3:     {roi_data.quantile(0.75):.4f}")
#         print(f"    Tracks: {len(roi_data)}")

# # Create comparison table
# print("\n\nComparison Table (Mean ± Std):")
# print("-" * 60)
# print(f"{config['xlabel']}\t" + "\t".join([f"DynROI {i+1}" for i in range(3)]))

# for param in avg_df['Parameter'].unique():
#     param_data = avg_df[avg_df['Parameter'] == param]
#     values = []
#     for roi in ['roi_001', 'roi_002', 'roi_003']:
#         roi_data = param_data[param_data['ROI'] == roi]['Avg_Trackability']
#         if len(roi_data) > 0:
#             values.append(f"{roi_data.mean():.3f}±{roi_data.std():.3f}")
#         else:
#             values.append("N/A")
#     print(f"{param}\t" + "\t".join(values))

In [ ]:
# tracking_sweep_data['msn_gap1_rad1']['roi_001']

In [ ]:
# def create_plot_data(trackability):
#     plot_data = []
    
#     # for param in ['msn_gap1_rad1', 'msn_gap2_rad1', 'msn_gap3_rad1']:
#     for param in ['msn_gap1_rad1', 'msn_gap1_rad2']:
#         for roi in ['roi_001', 'roi_002', 'roi_003']:
#             if (param, roi) in trackability.index:
#                 values = trackability.loc[param, roi].iloc[0]  # This is already the list
#                 gap_num = param.split('_')[1]  # Extract gap1, gap2, gap3
#                 rad_num = param.split('_')[2]  # Extract rad1 and rad2
                
#                 for value in values:
#                     plot_data.append({
#                         'Rad': rad_num,
#                         'Gap': gap_num,
#                         'Trackability': value,
#                         'ROI': roi
#                     })
    
#     return pd.DataFrame(plot_data)

# # Create the plot data
# plot_df = create_plot_data(trackability)

In [ ]:
# import matplotlib.pyplot as plt
# import seaborn as sns

# # Set style and color palette
# sns.set_style("whitegrid")
# plt.figure(figsize=(10, 7))

# # Create violin plot with better styling
# # ax = sns.violinplot(x="Gap", y="Trackability", data=plot_df, 
# #                     inner=None, color='lightblue', alpha=0.6,
# #                     linewidth=1.5, saturation=0.8)

# ax = sns.violinplot(x="Rad", y="Trackability", data=plot_df, 
#                     inner=None, color='lightblue', alpha=0.6,
#                     linewidth=1.5, saturation=0.8)

# # Calculate means for each ROI
# # means_df = plot_df.groupby(['Gap', 'ROI'])['Trackability'].mean().reset_index()
# means_df = plot_df.groupby(['Rad', 'ROI'])['Trackability'].mean().reset_index()

# # Create a custom color palette for ROIs
# roi_colors = ['#1f77b4', '#ff7f0e', '#2ca02c']  # Blue, Orange, Green
# roi_palette = dict(zip(['roi_001', 'roi_002', 'roi_003'], roi_colors))

# # Plot means as larger points with better styling
# sns.stripplot(x="Rad", y="Trackability", hue="ROI", data=means_df, 
#               size=12, dodge=False, jitter=True, palette=roi_palette,
#               edgecolor='white', linewidth=2, alpha=0.9)

# # Improve formatting
# plt.title('Minimum Radius Parameter Sweep\n(merge/split = no, gap_size = 1)', 
#           fontsize=16, fontweight='bold', pad=20)
# plt.ylabel('Trackability Score', fontsize=14, fontweight='semibold')
# plt.xlabel('Parameter Condition', fontsize=14, fontweight='semibold')

# # Improve legend
# handles, labels = ax.get_legend_handles_labels()
# new_labels = ['DynROI 1', 'DynROI 2', 'DynROI 3']
# plt.legend(handles, new_labels, title='Region of Interest', 
#            title_fontsize=12, fontsize=11, loc='lower right',
#            frameon=True, fancybox=True, shadow=True)

# # Set y-axis limits for better visibility
# plt.ylim(0, 1.05)

# # Improve x-axis labels
# plt.xticks(fontsize=12)
# plt.yticks(fontsize=11)

# # Add subtle grid
# plt.grid(True, alpha=0.3, linestyle='--')

# # Tight layout for better spacing
# plt.tight_layout()
# plt.show()

In [ ]:
# import matplotlib.pyplot as plt
# import seaborn as sns
# import numpy as np
# import pandas as pd

# # Set bin edges and labels
# bins = np.arange(-0.1, 1.15, 0.05)  # 0.05 bin size, upper limit 1.1 inclusive
# bin_labels = [f"({bins[i]:.2f}, {bins[i+1]:.2f}]" for i in range(len(bins)-1)]

# # Filter out NaN values
# clean_df = plot_df.dropna(subset=['Trackability']).round(4)

# # Set style
# sns.set_style("whitegrid")

# # ------------------ Part 1: Raw Histogram ------------------
# plt.figure(figsize=(10, 6))
# sns.histplot(data=clean_df, x='Trackability', bins=bins, hue='Rad',
#              multiple='dodge', shrink=0.85, palette='pastel', edgecolor='black')
# plt.title("Histogram of Trackability Scores by Minimum Radius", fontsize=15, fontweight='bold')
# plt.xlabel("Trackability Score", fontsize=13)
# plt.ylabel("Count", fontsize=13)
# plt.xticks(bins)
# plt.grid(True, linestyle='--', alpha=0.3)
# plt.tight_layout()
# plt.show()

In [ ]:
# clean_df = plot_df.dropna(subset=['Trackability']).copy()  # ← make an explicit copy
# # 2. Snap values to 4 decimal places to avoid floating point binning issues
# clean_df['Trackability'] = clean_df['Trackability'].round(4)

# clean_df.loc[:, 'bin'] = pd.cut(clean_df['Trackability'], bins=bins, labels=bin_labels, right=True)
    
# # Count per bin and gap
# bin_counts = clean_df.groupby(['Rad', 'bin'], observed=True).size().reset_index(name='count')

# # Total counts per gap for normalization
# gap_totals = bin_counts.groupby('Rad')['count'].transform('sum')
# bin_counts['percent'] = 100 * bin_counts['count'] / gap_totals

# plt.figure(figsize=(14, 6))
# ax = sns.barplot(data=bin_counts, x='bin', y='percent', hue='Rad', palette='pastel')

# # Title and labels
# ax.set_title("Percentage of Trackability Scores per Bin by Gap", fontsize=16, fontweight='bold', pad=15)
# ax.set_xlabel("Trackability Score Bin", fontsize=14, fontweight='semibold', labelpad=10)
# ax.set_ylabel("Percentage (%)", fontsize=14, fontweight='semibold', labelpad=10)

# # Tick formatting
# plt.xticks(rotation=90, fontsize=11)
# plt.yticks(fontsize=11)

# # Grid
# plt.grid(axis='y', linestyle='--', alpha=0.4)

# # Legend
# plt.legend(title='Gap Size', title_fontsize=12, fontsize=11, loc='upper left', frameon=True, shadow=False)

# # Layout
# plt.tight_layout()
# plt.show()

In [ ]:
# # Print number of segments per bin per Gap
# print("🔢 Segment Count per Bin per Gap:\n")
# for gap in bin_counts['Gap'].unique():
#     print(f"Gap = {gap}")
#     subset = bin_counts[bin_counts['Gap'] == gap][['bin', 'count']]
#     for _, row in subset.iterrows():
#         print(f"  Bin {row['bin']}: {int(row['count'])} segments")
#     total = subset['count'].sum()
#     print(f"  ➤ Total segments for Gap {gap}: {int(total)}\n")

# # Print total segment count across all gaps
# overall_total = bin_counts['count'].sum()
# print(f"✅ Overall total number of segments: {int(overall_total)}")

In [ ]:
base_dir = r'Z:\Abhi\LLSM_Analysis'
input_file_directory = '41_OS_analysis/'

zarr_file_directory = input_file_directory + 'zarr_file/all_channels_data'
zarr_full_path = os.path.join(base_dir, zarr_file_directory)


input_directory_trackability = os.path.join(base_dir, input_file_directory) + 'datasets/filtered_tracks_final_trackability.pkl'
input_directory_trackability_prefilter_expaned = os.path.join(base_dir, input_file_directory) + 'datasets/track_df_cleaned_final_trackability.pkl'
input_directory_trackability_prefilter = os.path.join(base_dir, input_file_directory) + 'datasets/prefilter_tracks_final_trackability.pkl'

input_directory = os.path.join(base_dir, input_file_directory) + 'datasets/filtered_tracks_final.pkl'
input_directory_prefilter_expanded = os.path.join(base_dir, input_file_directory) + 'datasets/track_df_cleaned_final_full.pkl'
input_directory_prefilter = os.path.join(base_dir, input_file_directory) + 'datasets/prefilter_tracks_final.pkl'

In [ ]:
trackability_df = pd.read_pickle(input_directory_trackability)
trackability_prefilter_df = pd.read_pickle(input_directory_trackability_prefilter)
trackability_prefilter_expanded_df = pd.read_pickle(input_directory_trackability_prefilter_expaned)

track_df = pd.read_pickle(input_directory)
track_prefilter_df = pd.read_pickle(input_directory_prefilter)
track_prefilter_expanded_df = pd.read_pickle(input_directory_prefilter_expanded)

In [ ]:
# trackability_prefilter_df[trackability_prefilter_df['track_id'] == 0]

In [ ]:
# # Sort trackability_prefilter_expanded_df by 'track_id' to ensure consistent order, and show head(5)
# sorted_df = trackability_prefilter_expanded_df.sort_values(by=['frame','track_id'])
# sorted_df[sorted_df['track_id'] == 0]['c1_voxel_sum_adjusted']

In [ ]:
#### Do the tracks with trackability (post filtering) match the tracks in the track_df (original tracking, post filtering)? ####

matches = []
no_matches = []
total_rows = len(trackability_df)

for idx, row in tqdm(trackability_df.iterrows(), desc="Processing tracks", total=total_rows):
    track_id = row['track_id']
    
    # Extract lists from trackability_df (current row)
    trackability_mu_x = row['mu_x'].reset_index(drop=True).tolist()
    trackability_mu_y = row['mu_y'].reset_index(drop=True).tolist()
    trackability_mu_z = row['mu_z'].reset_index(drop=True).tolist()
    
    # Check each row in track_df for matches
    match_found = False
    for _, track_row in track_df.iterrows():
        # Extract lists from track_df (current row)
        track_mu_x = track_row['mu_x'].reset_index(drop=True).tolist()
        track_mu_y = track_row['mu_y'].reset_index(drop=True).tolist()
        track_mu_z = track_row['mu_z'].reset_index(drop=True).tolist()
        
        # Compare the lists
        if (trackability_mu_x == track_mu_x and 
            trackability_mu_y == track_mu_y and 
            trackability_mu_z == track_mu_z):
            match_found = True
            break
    

    # Inside the loop, replace the print statement with:
    if match_found:
        matches.append((idx, track_id))
    else:
        no_matches.append((idx, track_id))

In [ ]:
# Plot matches/(mathes + no_matches) and no_matches/(matches + no_matches) on a bar plot
total_count = len(matches) + len(no_matches)
match_percentage = (len(matches) / total_count) * 100 if total_count > 0 else 0
no_match_percentage = (len(no_matches) / total_count) * 100 if total_count > 0 else 0
percentages = [match_percentage, no_match_percentage]
labels = ['Matches', 'No Matches']
plt.figure(figsize=(4, 3.5))
sns.barplot(x=labels, y=percentages, hue = labels, palette=['#4CAF50', '#F44336'], legend =False)
plt.title('Trackability Matches vs No Matches')
plt.ylabel('Percentage (%)')
plt.ylim(0, 100)
print(f"Total matches: {len(matches)}, Total no matches: {len(no_matches)}")
# print percentage of matches and no matches
print(f"Match percentage: {match_percentage:.2f}%, No match percentage: {no_match_percentage:.2f}%")

In [ ]:
# trackability_df[trackability_df['track_id'] == 194]


In [ ]:
# track_df[(track_df['track_length'] == 22) & (track_df['track_start'] == 98)& (track_df['track_end'] == 119)]

In [ ]:
#### Optional: Comparing tracks with trackability (pre-filtering) to the original track_df ####

matches_prefilter = []
no_matches_prefilter = []
total_prefilter_rows = len(trackability_prefilter_df)

for idx, row in tqdm(trackability_prefilter_df.iterrows(), desc="Processing tracks", total=total_prefilter_rows):
    prefilter_track_id = row['track_id']
    
    # Extract lists from trackability_df (current row)
    prefilter_trackability_mu_x = row['mu_x'].reset_index(drop=True).tolist()
    prefilter_trackability_mu_y = row['mu_y'].reset_index(drop=True).tolist()
    prefilter_trackability_mu_z = row['mu_z'].reset_index(drop=True).tolist()


   
   # Check each row in track_df for matches
    match_found = False
    for _, track_row in track_prefilter_df.iterrows():
       # Extract lists from track_df (current row)
       track_mu_x = track_row['mu_x'].reset_index(drop=True).tolist()
       track_mu_y = track_row['mu_y'].reset_index(drop=True).tolist()
       track_mu_z = track_row['mu_z'].reset_index(drop=True).tolist()
       
       # Compare the entire track lists
       if (prefilter_trackability_mu_x == track_mu_x and 
           prefilter_trackability_mu_y == track_mu_y and 
           prefilter_trackability_mu_z == track_mu_z):
           match_found = True
           break
   
    if match_found:
       matches_prefilter.append(int(track_id))
    else:
        no_matches_prefilter.append(int(track_id))

In [ ]:
# Plot matches/(mathes + no_matches) and no_matches/(matches + no_matches) on a bar plot
total_count = len(matches_prefilter) + len(no_matches_prefilter)
match_percentage = (len(matches_prefilter) / total_count) * 100 if total_count > 0 else 0
no_match_percentage = (len(no_matches_prefilter) / total_count) * 100 if total_count > 0 else 0
percentages = [match_percentage, no_match_percentage]
labels = ['Matches', 'No Matches']
plt.figure(figsize=(4, 3.5))
sns.barplot(x=labels, y=percentages, hue = labels, palette=['#4CAF50', '#F44336'], legend =False)
plt.title('Trackability Matches vs No Matches')
plt.ylabel('Percentage (%)')
plt.ylim(0, 100)
print(f"Total matches: {len(matches_prefilter)}, Total no matches: {len(no_matches_prefilter)}")
# print percentage of matches and no matches
print(f"Match percentage: {match_percentage:.2f}%, No match percentage: {no_match_percentage:.2f}%")

In [ ]:
#### Same calculation done with a different but equivalent dataframe for completeness. Not super important. ####

matches_prefilter = []
no_matches_prefilter = []

# Pre-compute all track data once (instead of inside nested loops)
trackability_tracks = {}
for track_id in trackability_prefilter_expanded_df['track_id'].unique():
    track_data = trackability_prefilter_expanded_df[trackability_prefilter_expanded_df['track_id'] == track_id]
    trackability_tracks[track_id] = (
        track_data['mu_x'].tolist(),
        track_data['mu_y'].tolist(), 
        track_data['mu_z'].tolist()
    )

track_tracks = {}
for track_id in track_prefilter_expanded_df['track_id'].unique():
    track_data = track_prefilter_expanded_df[track_prefilter_expanded_df['track_id'] == track_id]
    track_tracks[track_id] = (
        track_data['mu_x'].tolist(),
        track_data['mu_y'].tolist(),
        track_data['mu_z'].tolist()
    )

# Now do the comparison (much faster)
for track_id in tqdm(trackability_tracks.keys(), desc="Processing tracks"):
    trackability_coords = trackability_tracks[track_id]
    
    match_found = any(
        trackability_coords == track_coords 
        for track_coords in track_tracks.values()
    )
    
    if match_found:
        matches_prefilter.append(int(track_id))
    else:
        no_matches_prefilter.append(int(track_id))

print(f"Total matches: {len(matches_prefilter)}, Total no matches: {len(no_matches_prefilter)}")

In [ ]:
# # Sanity checking the track export and trackability scores
# trackability_prefilter_df[trackability_prefilter_df['track_id'] == 251]